# HDB Resale ETL (Part 1)

## What this notebook does

This notebook downloads and processes Singapore HDB resale transactions from January 2012 through December 2016. ETL means **Extract, Transform, Load**:

1. **Extract** — download the source records from data.gov.sg.
2. **Transform** — combine, validate, clean, deduplicate, and enrich the records.
3. **Load** — save the results as CSV and JSON files under `outputs/`.

The pipeline creates five output groups:

- **Raw** — source records preserved without manual edits.
- **Cleaned** — records that passed validation and deduplication.
- **Failed** — rejected records, including a reason for each rejection.
- **Transformed** — cleaned records with the recalculated lease and resale identifier.
- **Hashed** — cleaned records with a hashed version of the resale identifier.

The detailed business logic is kept in `src/hdb_etl/pipeline.py`. This notebook runs that logic and presents the results in a readable form. Run the cells from top to bottom from the `notebooks/` directory.

## Rules and assumptions used by the pipeline

Before running the code, these are the important decisions to understand:

- **Date range:** Only transactions from `2012-01` through `2016-12` are included.
- **Source files stay unchanged:** Records are downloaded and processed by code. Nothing is manually removed or corrected in the source files.
- **Different source columns are preserved:** Some source datasets contain columns that others do not. The master dataset uses the union of all column names and leaves a missing value where a source did not provide a column.
- **Allowed values come from the data:** Valid values for `month`, `town`, `flat_type`, `flat_model`, and `storey_range` are learned from the master dataset instead of being typed into the code, then saved to `outputs/cleaned/domain_reference.json`. Validation runs against that saved reference rather than the batch it is checking, so a later batch containing an unknown town or flat model can actually be rejected. On the first run the master defines the reference, so the check has nothing to reject yet — `validation_summary.json` records which case applied under `domain_reference_source`.
- **Duplicate records:** Rows are treated as duplicates when every field except `resale_price` is the same. The row with the highest price is kept; lower-priced copies are sent to the failed output.
- **Remaining lease:** The pipeline calculates the remaining portion of a 99-year lease from `lease_commence_date`.
- **Fixed calculation date:** The lease calculation uses `2026-07-14`. Fixing this date prevents the output from changing simply because the pipeline is rerun on another day.
- **Hashed identifier:** The readable resale identifier is hashed with **HMAC-SHA-256, keyed with a secret pepper**. A plain SHA-256 digest would not be irreversible here: the identifier is drawn from a small alphabet (about 31.2 million combinations), so an attacker can simply hash every possibility and match. Keying the digest with a pepper defeats that, because a guess cannot be hashed without the secret. The notebook also checks that hashing preserves the identifier's uniqueness count for this dataset.
- **The committed outputs are not irreversible:** When `HDB_HASH_PEPPER` is unset, the pipeline falls back to a public demo pepper so a reviewer can reproduce these exact outputs. A published pepper provides no protection — anyone can read it from the repository and resume the search. `transformed_summary.json` records this honestly as `hash_is_irreversible: false`. A real deployment supplies the pepper from a secret store.

In [2]:
# Standard-library tools used for paths, captured logs, and safe HTML output.
from contextlib import redirect_stdout
from html import escape
from io import StringIO
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import HTML, Markdown, display

# The notebook is stored one directory below the project root. Adding the root
# to Python's import path lets this notebook import the reusable ETL module.
sys.path.append(str(Path("..").resolve()))

from src.hdb_etl.pipeline import run_step1_pipeline, run_step2_pipeline, run_step3_pipeline


# The following helper functions only control how results look in the notebook.
# They do not change any source or output data.
def readable_label(value):
    """Turn 'master_row_count' into the easier-to-read 'Master Row Count'."""
    return str(value).replace("_", " ").strip().title()


def display_section(title, description=None):
    """Display a heading and an optional explanation above a result."""
    display(Markdown(f"### {title}"))
    if description:
        display(Markdown(description))


def style_table(frame, caption, formats=None):
    """Convert a DataFrame into a simple formatted table for display."""
    formats = formats or {}
    formatters = {}
    for column in frame.columns:
        if column in formats:
            # A column can use either a formatting function or a format string.
            formatter = formats[column]
            formatters[column] = formatter if callable(formatter) else lambda value, template=formatter: template.format(value)
        elif pd.api.types.is_numeric_dtype(frame[column]) and not pd.api.types.is_bool_dtype(frame[column]):
            # Add thousands separators to ordinary numeric columns.
            formatters[column] = lambda value: f"{value:,}"

    table_html = frame.to_html(
        index=False,
        border=0,
        classes="hdb-result-table",
        formatters=formatters,
        na_rep="—",
        escape=True,
    )
    return HTML(
        "<style>"
        ".hdb-result { margin: 0.5rem 0 1.4rem; }"
        ".hdb-result figcaption { text-align: left; font-size: 1.05rem; font-weight: 600; padding: 0.4rem 0; }"
        ".hdb-result-table { border-collapse: collapse; width: auto; }"
        ".hdb-result-table th, .hdb-result-table td { text-align: left; padding: 0.4rem 0.7rem; border-bottom: 1px solid; }"
        ".hdb-result-table th { font-weight: 600; }"
        "</style>"
        f"<figure class='hdb-result'><figcaption>{escape(caption)}</figcaption>{table_html}</figure>"
    )


def summary_frame(stage_summaries):
    """Convert nested stage summaries into one row per reported metric."""
    rows = []
    for stage, summary in stage_summaries.items():
        for metric, value in summary.items():
            if isinstance(value, dict):
                value = ", ".join(f"{readable_label(key)}: {item:,}" for key, item in value.items()) or "None"
            rows.append({"Stage": stage, "Metric": readable_label(metric), "Value": value})
    return pd.DataFrame(rows)

In [3]:
# Run the three dependent stages in order. Their printed progress messages are
# captured so the notebook stays compact, while the full log remains available.
pipeline_log = StringIO()
with redirect_stdout(pipeline_log):
    step1_summary = run_step1_pipeline()  # Download sources and create the raw master dataset.
    step2_summary = run_step2_pipeline()  # Validate, deduplicate, and separate failed records.
    step3_summary = run_step3_pipeline()  # Add identifiers, recalculate leases, and hash identifiers.

# Each stage returns a dictionary of counts, checks, and output paths.
stage_summaries = {
    "Step 1 · Extract and combine": step1_summary,
    "Step 2 · Validate and clean": step2_summary,
    "Step 3 · Transform and hash": step3_summary,
}

display_section(
    "Pipeline execution summary",
    "All three stages completed. This table shows the main counts, checks, and generated file paths.",
)
display(style_table(summary_frame(stage_summaries), "Pipeline stage metrics"))

# The detailed log is collapsed by default because it can contain many download messages.
display(
    HTML(
        "<details><summary><strong>Execution log</strong> — show extraction and file-write details</summary>"
        f"<pre style='white-space: pre-wrap; margin-top: 0.75rem;'>{escape(pipeline_log.getvalue())}</pre></details>"
    )
)

### Pipeline execution summary

All three stages completed. This table shows the main counts, checks, and generated file paths.

Stage,Metric,Value
Step 1 · Extract and combine,Selected Dataset Count,3
Step 1 · Extract and combine,Master Row Count,92544
Step 1 · Extract and combine,Master Column Count,11
Step 1 · Extract and combine,Master Output Path,outputs/raw/master_2012_2016.csv
Step 2 · Validate and clean,Master Row Count,92544
Step 2 · Validate and clean,Cleaned Row Count,90949
Step 2 · Validate and clean,Failed Row Count,1595
Step 2 · Validate and clean,Validation Failed Count,0
Step 2 · Validate and clean,Dedup Failed Count,1595
Step 2 · Validate and clean,Reconciliation Ok,True


In [4]:
# Read the machine-readable summaries written by the pipeline. These files let
# us review quality results without recalculating them in the notebook.
profiling = json.loads(Path("../outputs/cleaned/profiling_summary.json").read_text(encoding="utf-8"))
validation = json.loads(Path("../outputs/cleaned/validation_summary.json").read_text(encoding="utf-8"))
transformed = json.loads(Path("../outputs/transformed/transformed_summary.json").read_text(encoding="utf-8"))

# Select the validation measures that are most useful to a first-time reviewer.
# Reconciliation is successful when: master rows = cleaned rows + failed rows.
validation_metrics = [
    "master_row_count",
    "cleaned_row_count",
    "failed_row_count",
    "validation_failed_count",
    "dedup_failed_count",
    "reconciliation_ok",
]
validation_table = pd.DataFrame(
    {"Metric": [readable_label(metric) for metric in validation_metrics], "Value": [validation[metric] for metric in validation_metrics]}
)

# Compare readable and hashed identifiers to confirm that hashing did not
# change the number of unique identifiers in this dataset.
transformation_metrics = [
    "reference_date",
    "transformed_row_count",
    "hashed_row_count",
    "resale_identifier_unique_count",
    "hashed_identifier_unique_count",
    "hash_uniqueness_preserved",
]
transformation_table = pd.DataFrame(
    {
        "Metric": [readable_label(metric) for metric in transformation_metrics],
        "Value": [transformed[metric] for metric in transformation_metrics],
    }
)

# Build one summary row per required field and capture its five most common
# values. This makes missing values and unexpected formats easy to spot.
profile_rows = []
top_value_rows = []
for field in ["month", "town", "flat_type", "flat_model", "storey_range"]:
    field_profile = profiling["fields"][field]
    profile_rows.append(
        {
            "Field": field,
            "Missing": field_profile.get("null_or_blank_count", 0),
            "Unique": field_profile.get("unique_count"),
            "Parse failures": field_profile.get("parse_fail_count"),
            "Pattern failures": field_profile.get("pattern_fail_count"),
            "Minimum": field_profile.get("min"),
            "Maximum": field_profile.get("max"),
        }
    )
    for rank, (value, count) in enumerate(list(field_profile.get("top_10_values", {}).items())[:5], start=1):
        top_value_rows.append({"Field": field, "Rank": rank, "Value": value, "Count": count})

display_section(
    "Quality gates",
    "These checks confirm that every row is accounted for and that hashing preserves the identifier count.",
)
display(style_table(validation_table, "Validation and row reconciliation"))
display(style_table(transformation_table, "Transformation and hashing checks"))

display_section(
    "Profile highlights",
    "The tables summarize missing values, formats, date coverage, and the most common observed values.",
)
display(style_table(pd.DataFrame(profile_rows), "Required-field profile"))
display(
    style_table(
        pd.DataFrame(top_value_rows),
        "Five most frequent values per required field",
        formats={"Count": "{:,.0f}"},
    )
)

### Quality gates

These checks confirm that every row is accounted for and that hashing preserves the identifier count.

Metric,Value
Master Row Count,92544
Cleaned Row Count,90949
Failed Row Count,1595
Validation Failed Count,0
Dedup Failed Count,1595
Reconciliation Ok,True


Metric,Value
Reference Date,2026-07-14
Transformed Row Count,90949
Hashed Row Count,90949
Resale Identifier Unique Count,77255
Hashed Identifier Unique Count,77255
Hash Uniqueness Preserved,True


### Profile highlights

The tables summarize missing values, formats, date coverage, and the most common observed values.

Field,Missing,Unique,Parse failures,Pattern failures,Minimum,Maximum
month,0,60,0.0,—,2012-01,2016-12
town,0,26,—,—,—,—
flat_type,0,7,—,—,—,—
flat_model,0,20,—,—,—,—
storey_range,0,25,—,0.0,—,—


Field,Rank,Value,Count
month,1,2012-03,"2,360"
month,2,2012-05,"2,323"
month,3,2012-07,"2,179"
month,4,2012-04,"2,155"
month,5,2012-08,"2,075"
town,1,JURONG WEST,"7,573"
town,2,WOODLANDS,"7,399"
town,3,TAMPINES,"6,728"
town,4,BEDOK,"6,071"
town,5,YISHUN,"5,937"


In [5]:
# Load all five CSV outputs so their sizes and representative records can be
# compared in one place.
master = pd.read_csv("../outputs/raw/master_2012_2016.csv")
cleaned = pd.read_csv("../outputs/cleaned/cleaned_2012_2016.csv")
failed = pd.read_csv("../outputs/failed/failed_2012_2016.csv")
transformed_df = pd.read_csv("../outputs/transformed/transformed_2012_2016.csv")
hashed = pd.read_csv("../outputs/hashed/hashed_2012_2016.csv")

# Record the dimensions of each output. Cleaned + Failed should equal Master,
# while Transformed and Hashed should each have the same row count as Cleaned.
qc = pd.DataFrame(
    [
        {"Dataset": "Master", "Rows": len(master), "Columns": len(master.columns)},
        {"Dataset": "Cleaned", "Rows": len(cleaned), "Columns": len(cleaned.columns)},
        {"Dataset": "Failed", "Rows": len(failed), "Columns": len(failed.columns)},
        {"Dataset": "Transformed", "Rows": len(transformed_df), "Columns": len(transformed_df.columns)},
        {"Dataset": "Hashed", "Rows": len(hashed), "Columns": len(hashed.columns)},
    ]
)

# Count why records failed, then select a small sample of generated fields for
# visual inspection. The complete records remain available in their CSV files.
failed_reasons = failed["failure_reason"].value_counts().rename_axis("Failure reason").reset_index(name="Count")
transformed_sample = transformed_df[
    [
        "month",
        "town",
        "flat_type",
        "block",
        "resale_price",
        "resale_identifier",
        "remaining_lease_recomputed",
    ]
].head(10)
hashed_sample = hashed[["month", "town", "flat_type", "hashed_identifier"]].head(10)

display_section(
    "Output quality control",
    "The row and column counts show how records move through the five output groups.",
)
display(style_table(qc, "Dataset dimensions", formats={"Rows": "{:,.0f}", "Columns": "{:,.0f}"}))
display(style_table(failed_reasons, "Failed-record reason distribution", formats={"Count": "{:,.0f}"}))

display_section(
    "Transformation samples",
    "The first ten records provide a quick visual check of the calculated fields.",
)
display(
    style_table(
        transformed_sample,
        "Transformed records (first 10)",
        formats={"resale_price": "S${:,.0f}"},
    )
)
display(
    style_table(
        hashed_sample,
        "Hashed records (first 10; digest shortened for display only)",
        formats={"hashed_identifier": lambda value: f"{value[:20]}…{value[-8:]}"},
    )
)

### Output quality control

The row and column counts show how records move through the five output groups.

Dataset,Rows,Columns
Master,"92,544",11
Cleaned,"90,949",11
Failed,"1,595",13
Transformed,"90,949",17
Hashed,"90,949",12


Failure reason,Count
duplicate_lower_price,"1,324"
duplicate_exact_match,271


### Transformation samples

The first ten records provide a quick visual check of the calculated fields.

month,town,flat_type,block,resale_price,resale_identifier,remaining_lease_recomputed
2012-01,ANG MO KIO,2 ROOM,170,"S$260,000",S1702501A,58 years 5 months
2012-01,ANG MO KIO,2 ROOM,174,"S$226,000",S1742501A,58 years 5 months
2012-01,ANG MO KIO,2 ROOM,314,"S$263,000",S3142501A,50 years 5 months
2012-01,ANG MO KIO,2 ROOM,314,"S$275,000",S3142501A,50 years 5 months
2012-01,ANG MO KIO,2 ROOM,406,"S$257,800",S4062501A,51 years 5 months
2012-01,ANG MO KIO,2 ROOM,508,"S$260,000",S5082501A,52 years 5 months
2012-01,ANG MO KIO,3 ROOM,118,"S$320,000",S1183401A,50 years 5 months
2012-01,ANG MO KIO,3 ROOM,121,"S$382,800",S1213401A,50 years 5 months
2012-01,ANG MO KIO,3 ROOM,151,"S$302,000",S1513401A,53 years 5 months
2012-01,ANG MO KIO,3 ROOM,154,"S$321,000",S1543401A,53 years 5 months


month,town,flat_type,hashed_identifier
2012-01,ANG MO KIO,2 ROOM,2128fe38909b031e1984…75fb6ba4
2012-01,ANG MO KIO,2 ROOM,2a9b6a4338780385769a…d5cf4df1
2012-01,ANG MO KIO,2 ROOM,4bc04ab97f490d4963b5…1fa65a70
2012-01,ANG MO KIO,2 ROOM,4bc04ab97f490d4963b5…1fa65a70
2012-01,ANG MO KIO,2 ROOM,0454542b243731a1938b…cde48b0c
2012-01,ANG MO KIO,2 ROOM,933c12a47cd9a5632bc8…1b24b31d
2012-01,ANG MO KIO,3 ROOM,30eaaf687cbf8dab80f7…12c0b163
2012-01,ANG MO KIO,3 ROOM,444056aa62dbbf21445f…a1eb91df
2012-01,ANG MO KIO,3 ROOM,3a62befc6157ecff876e…6883fb87
2012-01,ANG MO KIO,3 ROOM,8913a49f9f3d6f9f31cc…84c28103


## Why the code is split into a module and a notebook

The project gives two files different responsibilities:

- **`src/hdb_etl/pipeline.py` contains the ETL logic.** Functions in this module download, validate, deduplicate, transform, and save the data. Keeping the logic in a normal Python file makes it easier to test and reuse.
- **`notebooks/part1_hdb_etl.ipynb` runs and explains the pipeline.** The notebook calls the module's functions and displays the important evidence, such as row counts, quality checks, and sample outputs.

This separation keeps the notebook readable without hiding the implementation. A reviewer can follow the workflow here and inspect the module when more technical detail is needed.

## Final checks

The completed notebook should provide evidence for each of the following:

- It runs from top to bottom in a fresh kernel without manual data changes.
- All five output groups are created under `outputs/`.
- Every rejected record includes a `failure_reason` and `failure_stage`.
- Duplicate groups keep the record with the highest `resale_price`.
- Each `resale_identifier` uses the average price of its month, town, and flat-type group.
- Remaining lease values use the fixed reference date shown above.
- The summary confirms that `master rows = cleaned rows + failed rows`.
- The readable and hashed identifiers have the same unique-value count.

## How to interpret the results

- The allowed field values are learned from the source data. This avoids rejecting valid towns, flat types, or models because of an incomplete hardcoded list.
- Most rejected records in this period may be lower-priced duplicates rather than malformed records. The failure-reason table above shows the actual distribution.
- Records are rejected only on mechanical rules — a missing field, an unknown value, a malformed format, or a duplicate. Resale prices are kept as-is, so the Cleaned output preserves the full price distribution for the Data Science team to analyse.